In [3]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Mar 14 10:11:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install transformers

In [ ]:
HF_TOKEN = "Your Hugging Face API token here"

In [6]:
import os

In [7]:
os.environ["HF_TOKEN"] = HF_TOKEN

In [8]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [10]:
model_name = "google/gemma-3-1b-it"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [12]:
tokenizer("Hello World")

{'input_ids': [2, 9259, 4109], 'attention_mask': [1, 1, 1]}

In [13]:
input_conversation = [
    { "role": "user", "content": "which is the best place to learn GenAI?" },
    { "role": "assistant", "content": "The best place to learn AI is "}
]

In [14]:
input_tokens = tokenizer.apply_chat_template(
    conversation=input_conversation,
    tokenize=True,

    #if tokenize=True then it will show in number.
)
input_tokens

{'input_ids': [2, 105, 2364, 107, 7650, 563, 506, 1791, 1977, 531, 3449, 8471, 12553, 236881, 106, 107, 105, 4368, 107, 818, 1791, 1977, 531, 3449, 12498, 563, 106, 107], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [15]:
input_detokens = tokenizer.apply_chat_template(
    conversation=input_conversation,
    tokenize=False,
    continue_final_message=True,

    #if tokenize=True then it will show in number.
)
input_detokens

'<bos><start_of_turn>user\nwhich is the best place to learn GenAI?<end_of_turn>\n<start_of_turn>model\nThe best place to learn AI is'

In [16]:
output_label = "GenAI Cohort by Suman"
full_conversation = input_detokens + output_label + tokenizer.eos_token
full_conversation

'<bos><start_of_turn>user\nwhich is the best place to learn GenAI?<end_of_turn>\n<start_of_turn>model\nThe best place to learn AI isGenAI Cohort by Suman<eos>'

In [17]:
input_tokenized = tokenizer(full_conversation, return_tensors="pt", add_special_tokens=False).to(device)["input_ids"]
input_tokenized

tensor([[     2,    105,   2364,    107,   7650,    563,    506,   1791,   1977,
            531,   3449,   8471,  12553, 236881,    106,    107,    105,   4368,
            107,    818,   1791,   1977,    531,   3449,  12498,    563,  14696,
          12553, 105657,    632,    684,    555,   6688,      1]],
       device='cuda:0')

In [18]:
input_ids = input_tokenized[:, :-1].to(device)
target_ids = input_tokenized[:, 1:].to(device)
print(f"input_ids: {input_ids}")
print(f"target_ids: {target_ids}")

input_ids: tensor([[     2,    105,   2364,    107,   7650,    563,    506,   1791,   1977,
            531,   3449,   8471,  12553, 236881,    106,    107,    105,   4368,
            107,    818,   1791,   1977,    531,   3449,  12498,    563,  14696,
          12553, 105657,    632,    684,    555,   6688]], device='cuda:0')
target_ids: tensor([[   105,   2364,    107,   7650,    563,    506,   1791,   1977,    531,
           3449,   8471,  12553, 236881,    106,    107,    105,   4368,    107,
            818,   1791,   1977,    531,   3449,  12498,    563,  14696,  12553,
         105657,    632,    684,    555,   6688,      1]], device='cuda:0')


In [19]:
import torch.nn as nn
def calculate_loss(logits, labels):
  loss_fn = nn.CrossEntropyLoss(reduction="none")
  cross_entropy = loss_fn(logits.view(-1, logits.shape[-1]), labels.view(-1))
  return cross_entropy

In [22]:
# pulling the model

import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16
).to(device)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

In [27]:
from torch.optim import AdamW
model.train()

optimizer = AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)

for _ in range(10):
  out = model(input_ids=input_ids)
  loss = calculate_loss(out.logits, target_ids).mean()
  #backward propagation
  loss.backward()
  optimizer.step()
  optimizer.zero_grad()
  print(loss.item())

9.760260581970215e-07
0.0001316070556640625
3.762543201446533e-07
1.385807991027832e-06
5.052424967288971e-08
1.4062970876693726e-07
6.51925802230835e-08
1.0826624929904938e-08
3.6088749766349792e-09
3.6088749766349792e-09


In [36]:
input_prompt = [
    { "role": "user", "content": "which is the best place to learn GenAi?" }
]
input = tokenizer.apply_chat_template(
    conversation=input_prompt,
    return_tensors='pt',
    tokenize=True,
).to(device)
output = model.generate(input, max_new_tokens=25)
print(tokenizer.batch_decode(output, skip_special_tokens=True))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (3750660839.py, line 7)

In [38]:
input_prompt = [
    {"role": "user", "content": "which is the best place to learn GenAI?"}
]

inputs = tokenizer.apply_chat_template(
    conversation=input_prompt,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

# move tensor to model device
inputs = inputs.to(device)

# Pass the unpacked inputs (input_ids and attention_mask) to model.generate
output = model.generate(**inputs, max_new_tokens=25)

print(tokenizer.batch_decode(output, skip_special_tokens=True))

['user\nwhich is the best place to learn GenAI?\nmodel\nThe best place to learn AI isGenAI Cohort by Suman']
